# Minecraft 攻略 Agent RAG

基于 RAG（检索增强生成）+ Agent（多步推理）的 Minecraft 中文攻略问答系统。

简单问题：`Router → RAG（检索 → 重排序 → DeepSeek 生成）`
复杂问题：`Router → Planner（拆解子问题）→ 多路 RAG → Synthesizer（综合生成）`


In [1]:
# ============================================================
# Cell 1: 文档加载 & 智能切分
# 从 docs/ 读取所有 .md 文件，按 Markdown 标题切分 chunk
# ============================================================
from pathlib import Path
from typing import List
import re

DOCS_DIR = "docs"

# 切分参数（可按需调整）
MIN_CHUNK_SIZE = 100   # 最短 chunk（小于此长度与下一 chunk 合并）
MAX_CHUNK_SIZE = 800   # 最长 chunk（超过此长度按段落二次切分）


def load_docs(docs_dir: str = DOCS_DIR) -> dict:
    """读取 docs/ 下所有 .md 文件，返回 {文件名: 内容} 字典"""
    docs = {}
    for filepath in Path(docs_dir).glob("*.md"):
        with open(filepath, "r", encoding="utf-8") as f:
            docs[filepath.stem] = f.read()
    return docs


def split_by_headings(text: str) -> List[str]:
    """
    按 Markdown 标题（## / ###）切分文档。
    策略：
      - 保留标题在 chunk 开头（给后续检索提供上下文）
      - 短 chunk（< MIN_CHUNK_SIZE）与下一个合并
      - 长 chunk（> MAX_CHUNK_SIZE）按空行二次切分
    """
    lines = text.split("\n")

    # 第一步：按 ## 或 ### 标题切分
    raw_chunks = []
    current = []
    for line in lines:
        if re.match(r"^#{2,3}\s", line) and current:
            raw_chunks.append("\n".join(current))
            current = [line]
        else:
            current.append(line)
    if current:
        raw_chunks.append("\n".join(current))

    # 过滤纯空白片段
    raw_chunks = [c.strip() for c in raw_chunks if c.strip()]

    # 第二步：合并短 chunk
    merged = []
    for chunk in raw_chunks:
        if merged and len(chunk) < MIN_CHUNK_SIZE:
            merged[-1] += "\n\n" + chunk
        else:
            merged.append(chunk)

    # 第三步：切分过长 chunk（按 \n\n 段落切）
    final = []
    for chunk in merged:
        if len(chunk) <= MAX_CHUNK_SIZE:
            final.append(chunk)
        else:
            paragraphs = chunk.split("\n\n")
            buffer = []
            for para in paragraphs:
                candidate = "\n\n".join(buffer + [para]) if buffer else para
                if len(candidate) > MAX_CHUNK_SIZE and buffer:
                    final.append("\n\n".join(buffer))
                    buffer = [para]
                else:
                    buffer.append(para)
            if buffer:
                final.append("\n\n".join(buffer))

    return final


def build_chunks_from_docs(docs_dir: str = DOCS_DIR) -> List[str]:
    """加载所有文档并按标题切分，返回全部 chunk 列表"""
    all_chunks = []
    docs = load_docs(docs_dir)
    print(f"从 {docs_dir}/ 加载了 {len(docs)} 个文档\n")
    print("切分结果:")
    for doc_name, content in docs.items():
        chunks = split_by_headings(content)
        all_chunks.extend(chunks)
        print(f"  {doc_name:20s} -> {len(chunks)} 个 chunk")
    print(f"  {'─' * 20}-> ─")
    print(f"  {'总计':20s} -> {len(all_chunks)} 个 chunk")
    return all_chunks


# 执行加载 & 切分
chunks = build_chunks_from_docs()

# 预览前 3 个 chunk
print("\n--- 预览前 3 个 chunk ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[chunk {i}] ({len(chunk)} 字符)")
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)

从 docs/ 加载了 29 个文档

切分结果:
  animal               -> 3 个 chunk
  armor                -> 4 个 chunk
  beacon               -> 3 个 chunk
  bed                  -> 2 个 chunk
  biome                -> 3 个 chunk
  bow                  -> 4 个 chunk
  brewing              -> 3 个 chunk
  combat               -> 4 个 chunk
  crafting             -> 4 个 chunk
  enchanting           -> 5 个 chunk
  experience           -> 4 个 chunk
  food                 -> 3 个 chunk
  hopper               -> 3 个 chunk
  mining               -> 3 个 chunk
  monster              -> 6 个 chunk
  nether               -> 5 个 chunk
  ore                  -> 4 个 chunk
  piston               -> 3 个 chunk
  potion               -> 3 个 chunk
  redstone             -> 5 个 chunk
  smelting             -> 3 个 chunk
  smithing             -> 4 个 chunk
  structure            -> 2 个 chunk
  sword                -> 3 个 chunk
  the_end              -> 4 个 chunk
  tools                -> 2 个 chunk
  trading              -> 3 个 chunk
  

In [2]:
# ============================================================
# Cell 2: 加载中文向量化模型
# ============================================================
from sentence_transformers import SentenceTransformer

# 中文句子编码器，输出 768 维向量
embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")


def embed_chunk(chunk: str) -> List[float]:
    """将文本转为向量"""
    embedding = embedding_model.encode(chunk)
    return embedding.tolist()


# 测试：打印向量维度
test_embedding = embed_chunk("Minecraft 攻略")
print(f"向量维度: {len(test_embedding)}")
print(f"前 5 维: {test_embedding[:5]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

向量维度: 768
前 5 维: [-0.8085852861404419, -0.7096291780471802, 0.5697712898254395, -0.4742099642753601, 0.4108605980873108]


In [3]:
# ============================================================
# Cell 3: 批量向量化 & 写入 ChromaDB
# ============================================================
import chromadb

# 创建内存向量数据库（重启即丢失，无需外部服务）
chromadb_client = chromadb.EphemeralClient()
chromadb_collection = chromadb_client.get_or_create_collection(name="minecraft")


def save_embeddings(chunks: List[str]) -> None:
    """将 chunk 批量向量化并写入 ChromaDB"""
    print(f"正在向量化 {len(chunks)} 个 chunk...")

    # 批量生成向量
    embeddings = []
    for i, chunk in enumerate(chunks):
        vec = embed_chunk(chunk)
        embeddings.append(vec)
        if (i + 1) % 50 == 0:
            print(f"  已完成 {i + 1}/{len(chunks)}")

    # 写入 ChromaDB
    ids = [str(i) for i in range(len(chunks))]
    chromadb_collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids
    )
    print(f"✓ 已写入 {len(chunks)} 个向量到 ChromaDB (集合: minecraft)")


# 执行入库
save_embeddings(chunks)

正在向量化 102 个 chunk...
  已完成 50/102
  已完成 100/102
✓ 已写入 102 个向量到 ChromaDB (集合: minecraft)


In [4]:
# ============================================================
# Cell 4: 检索
# 对用户查询做向量化 → ChromaDB 相似度搜索 → 返回 top_k 文本
# ============================================================
RETRIEVE_TOP_K = 10  # 召回数量（重排序会进一步筛选）


def retrieve(query: str, top_k: int = RETRIEVE_TOP_K) -> List[str]:
    """向量检索：查询 → 余弦相似度搜索 → 返回 top_k 个 chunk"""
    query_embedding = embed_chunk(query)
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results["documents"][0]


# 测试检索
test_query = "铁镐怎么做？"
retrieved_chunks = retrieve(test_query)

print(f"查询: {test_query}")
print(f"检索到 {len(retrieved_chunks)} 个 chunk:\n")
for i, chunk in enumerate(retrieved_chunks):
    preview = chunk[:150].replace("\n", " ")
    print(f"  [{i}] {preview}...")

查询: 铁镐怎么做？
检索到 10 个 chunk:

  [0] ### 镐（Pickaxe）  用于挖掘石头类方块、矿石等。采集矿石时必须使用正确材质的镐。  **合成配方**：3 个材料横排 + 2 根木棍竖排  **效率**：木镐 < 石镐 < 铁镐 < 金镐 < 钻石镐 < 下界合金镐  - **木镐**：可采石煤、圆石。耐久度低 - **石镐**：可采铁...
  [1] ## 剑  剑是 Minecraft 中的主要近战武器。  ### 合成配方  1 个木棍 + 2 个材料竖排放置（材料在顶部，木棍在底部）。...
  [2] ### 常见烧炼配方  | 输入 | 输出 | 用途 | |------|------|------| | 铁矿石 | 铁锭 | 制作工具、盔甲 | | 金矿石 | 金锭 | 制作工具、盔甲、动力铁轨 | | 铜矿石 | 铜锭 | 制作避雷针、望远镜 | | 沙子 | 玻璃 | 建筑、玻璃瓶 | |...
  [3] ### 主世界矿石  | 矿石 | 冶炼产物 | 主要用途 | |------|---------|---------| | 煤矿石 | 煤炭 | 燃料、火把 | | 铁矿石 | 铁锭 | 工具、盔甲、铁轨、漏斗等 | | 铜矿石 | 铜锭 | 避雷针、望远镜、铜块 | | 金矿石 | 金锭 | 动...
  [4] ## 漏斗  漏斗是用于传输物品的红石元件，可以朝下或侧面输送物品。  ### 合成配方  5 个铁锭 + 1 个箱子（铁锭摆 V 字型，底部中间放箱子）...
  [5] ## 活塞  活塞是红石驱动的机械元件，可以推动方块。  ### 普通活塞  **合成配方**：3 个木板 + 4 个圆石 + 1 个铁锭 + 1 个红石粉  **功能**：收到红石信号时推出活塞臂，推动最多 12 个方块。  ### 粘性活塞  **合成配方**：1 个普通活塞 + 1 个粘液球 ...
  [6] ### 经验获取方式  **采矿**：挖矿（尤其是煤矿、红石矿、青金石矿、下界石英矿）会掉落经验球  **刷怪**：击杀生物会掉落经验，Boss 掉落最多： - 末影龙：12,000 经验（首次击杀） - 凋零：50 经验  **熔炼**：从熔炉取出成品时获得经验  **繁殖**：让动物繁殖获得经验...
  [7

In [5]:
# ============================================================
# Cell 5: 重排序
# 使用交叉编码器对检索结果重新打分排序，提升准确率
# 注意：首次运行会自动下载模型（~200MB）
# ============================================================
from sentence_transformers import CrossEncoder

RERANK_TOP_K = 5  # 重排序后保留的数量

# 多语言交叉编码器：对 (查询, 文档) 对打分，比向量检索更精确但更慢
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")


def rerank(query: str, retrieved_chunks: List[str], top_k: int = RERANK_TOP_K) -> List[str]:
    """交叉编码器重排序：对候选 chunk 逐一打分，返回 top_k 最相关的"""
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)

    # 按分数降序排序
    ranked = sorted(
        zip(retrieved_chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # 打印每个 chunk 的分数（便于分析）
    for chunk, score in ranked:
        preview = chunk[:80].replace("\n", " ")
        print(f"    score={score:.4f}  {preview}...")

    return [chunk for chunk, _ in ranked[:top_k]]


# 测试重排序
print(f"重排序 ({len(retrieved_chunks)} 个候选 → top {RERANK_TOP_K}):")
reranked_chunks = rerank(test_query, retrieved_chunks)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

重排序 (10 个候选 → top 5):
    score=-2.3997  ### 镐（Pickaxe）  用于挖掘石头类方块、矿石等。采集矿石时必须使用正确材质的镐。  **合成配方**：3 个材料横排 + 2 根木棍竖排  **效率...
    score=-2.7235  ## 漏斗  漏斗是用于传输物品的红石元件，可以朝下或侧面输送物品。  ### 合成配方  5 个铁锭 + 1 个箱子（铁锭摆 V 字型，底部中间放箱子）...
    score=-3.4634  ## 锻造台与下界合金装备升级  锻造台是用于将钻石装备升级为下界合金装备的工作站方块。  ### 锻造台  **合成配方**：4 个木板 + 2 个铁锭  *...
    score=-3.6653  ## 活塞  活塞是红石驱动的机械元件，可以推动方块。  ### 普通活塞  **合成配方**：3 个木板 + 4 个圆石 + 1 个铁锭 + 1 个红石粉  ...
    score=-4.4086  ### 实用漏斗应用  **自动熔炉**：上面漏斗输入生食/矿石，侧面漏斗输入燃料，下面漏斗收集成品  **物品分类机**：漏斗 + 比较器。用命名物品作为过滤...
    score=-4.7714  ### 常用合成材料  - **木板**：原木在任意合成网格中放入 1 个 → 4 个木板 - **木棍**：2 个木板上下排列 → 4 个木棍 - **工作台...
    score=-5.0684  ## 剑  剑是 Minecraft 中的主要近战武器。  ### 合成配方  1 个木棍 + 2 个材料竖排放置（材料在顶部，木棍在底部）。...
    score=-5.1715  ### 经验获取方式  **采矿**：挖矿（尤其是煤矿、红石矿、青金石矿、下界石英矿）会掉落经验球  **刷怪**：击杀生物会掉落经验，Boss 掉落最多： -...
    score=-5.8312  ### 常见烧炼配方  | 输入 | 输出 | 用途 | |------|------|------| | 铁矿石 | 铁锭 | 制作工具、盔甲 | | 金矿石...
    score=-9.1866  ### 主世界矿石  | 矿石 | 冶炼产物 | 主要用途 | |------|----

In [6]:
# ============================================================
# Cell 6: DeepSeek 生成
# 将重排序后的 chunk 作为参考上下文，让 LLM 生成答案
# ============================================================
from dotenv import load_dotenv
from openai import OpenAI
import os

# 加载 .env 中的 API Key
load_dotenv()

# 初始化 DeepSeek 客户端（兼容 OpenAI 格式）
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

LLM_MODEL = "deepseek-v4-flash"
LLM_TEMPERATURE = 0.1  # 低温 → 事实性优先，适合 RAG


def generate(query: str, chunks: List[str]) -> str:
    """
    根据用户查询 + 参考 chunk，调用 DeepSeek 生成回答。
    要求模型严格基于参考内容作答，不编造。
    """
    prompt = f"""你是一位 Minecraft 中文攻略助手，请根据用户的问题和下列参考内容生成准确的回答。

用户问题: {query}

参考内容:
{"\n\n---\n\n".join(chunks)}

请基于上述参考内容作答，不要编造信息。如果参考内容不足以回答，请说明已知信息不足。"""

    print(f">>> 查询: {query}\n")

    response = deepseek_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=LLM_TEMPERATURE
    )

    return response.choices[0].message.content


# 单次测试
answer = generate(test_query, reranked_chunks)
print(f"\n回答:\n{answer}")

>>> 查询: 铁镐怎么做？


回答:
根据参考内容，铁镐的合成配方为：**3个铁锭横排 + 2根木棍竖排**。具体合成时，在工作台中，将3个铁锭放在第一行横排，然后将2根木棍分别放在第二行和第三行的中间一格（即竖排）。这样即可合成铁镐。铁镐可用于挖掘钻石矿石、红石矿石、金矿石和绿宝石矿石。


In [7]:
# ============================================================
# Cell 7: Agent — 多步推理层（Router / Planner / Executor / Synthesizer）
# 复杂问题：Router 判断 → Planner 拆解 → 多路检索 → 综合生成
# 简单问题：直接走现有单轮 RAG
# ============================================================
import json


def route_question(query: str) -> str:
    """LLM 判断问题复杂度，返回 simple 或 complex"""
    prompt = f"""你是一个 Minecraft 攻略问答系统的路由判断器。
判断以下问题属于 simple 还是 complex。

simple = 单一知识点，一步检索即可回答（如"铁镐怎么做"、"红石中继器作用"）
complex = 多步骤、多主题，需要拆解为多个子问题（如"从零做下界合金装备"）

仅回答 simple 或 complex，不要输出其他内容。

问题：{query}"""

    response = deepseek_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    result = response.choices[0].message.content.strip().lower()
    return result if result in ("simple", "complex") else "simple"


def plan_sub_questions(query: str) -> List[str]:
    """LLM 将复杂问题拆解为独立子问题列表"""
    prompt = f"""将以下复杂问题拆解为 2-4 个独立的子问题。

要求：
- 每个子问题应只涉及一个知识点，能通过一次攻略检索找到答案
- 子问题之间尽量独立，没有执行依赖
- 用中文

以 JSON 字符串数组格式输出，例如：
["子问题1", "子问题2", "子问题3"]

复杂问题：{query}"""

    response = deepseek_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    content = response.choices[0].message.content.strip()
    content = content.replace("```json", "").replace("```", "").strip()

    try:
        result = json.loads(content)
        if isinstance(result, list) and len(result) > 0:
            return result
        if isinstance(result, dict):
            for v in result.values():
                if isinstance(v, list) and len(v) > 0:
                    return v
    except json.JSONDecodeError:
        pass

    print("  ⚠ 子问题解析失败，回退为原问题")
    return [query]


def execute_plan(sub_questions: List[str]) -> List[str]:
    """对每个子问题执行检索+重排序，合并去重"""
    all_chunks = []
    seen = set()

    for i, q in enumerate(sub_questions, 1):
        print(f"  [{i}/{len(sub_questions)}] {q}")
        retrieved = retrieve(q)
        reranked = rerank(q, retrieved)
        for chunk in reranked:
            if chunk not in seen:
                seen.add(chunk)
                all_chunks.append(chunk)

    print(f"  → 合并后 {len(all_chunks)} 个唯一 chunk")
    return all_chunks


def agent_answer(query: str) -> str:
    """
    Agent 统一入口。
    simple → 单轮 RAG
    complex → 拆解 → 多路检索 → 综合生成
    """
    print(">>> 问题:", query)
    print()

    # Step 1: 路由
    print(">> 路由判断...", end=" ")
    route = route_question(query)
    print(route)

    if route == "simple":
        retrieved = retrieve(query)
        reranked = rerank(query, retrieved)
        return generate(query, reranked)

    # Step 2-4: Plan → Execute → Synthesize
    print(">> 拆解子问题...")
    sub_questions = plan_sub_questions(query)
    print(f"   → {sub_questions}")

    print(">> 多路检索...")
    all_chunks = execute_plan(sub_questions)

    print(">> 综合生成...")
    return generate(query, all_chunks)


print("Agent 模块加载完成 ✓")


Agent 模块加载完成 ✓


In [8]:
# ============================================================
# Cell 8: 完整测试套件 — 简单 + 复杂问题
# 简单问题验证 Router 正确路由到单轮 RAG
# 复杂问题验证 Planner → 多路检索 → Synthesizer 全流程
# ============================================================
TEST_QUESTIONS = [
    # ── 简单问题（应走 simple 路由） ──
    # 合成类
    "铁镐怎么做？需要哪些材料？",
    "怎么合成附魔台？",
    # 流程类
    "末地传送门怎么激活？",
    "下界合金锭的合成配方是什么？",
    # 机制类
    "附魔台最高能附到多少级？怎么提高等级上限？",
    "红石中继器有什么作用？",
    # 排查类
    "为什么我的地狱门没生成？怎么正确搭建？",

    # ── 复杂问题（应走 complex 路由） ──
    "从零开始做一套下界合金装备，需要经过哪些步骤？",
    "怎么建立一个自动熔炉系统？",
    "如何准备去打末影龙？从装备到附魔到战斗策略",
]


def run_test_suite(questions):
    """逐个问题跑 agent_answer，输出结果"""
    print("=" * 70)
    print("Agent + RAG 测试套件")
    print(f"共 {len(questions)} 个问题（简单 + 复杂）")
    print("=" * 70)

    for i, q in enumerate(questions, 1):
        print(f"\n{'─' * 70}")
        print(f"问题 [{i}/{len(questions)}]")
        print(f"{'─' * 70}")

        answer = agent_answer(q)

        print(f"\n>> 回答:")
        print(answer)
        print()

    print("=" * 70)
    print("全部测试完成")
    print("=" * 70)


run_test_suite(TEST_QUESTIONS)


Agent + RAG 测试套件
共 10 个问题（简单 + 复杂）

──────────────────────────────────────────────────────────────────────
问题 [1/10]
──────────────────────────────────────────────────────────────────────
>>> 问题: 铁镐怎么做？需要哪些材料？

>> 路由判断... simple
    score=-1.4094  ### 镐（Pickaxe）  用于挖掘石头类方块、矿石等。采集矿石时必须使用正确材质的镐。  **合成配方**：3 个材料横排 + 2 根木棍竖排  **效率...
    score=-3.3883  ### 常用合成材料  - **木板**：原木在任意合成网格中放入 1 个 → 4 个木板 - **木棍**：2 个木板上下排列 → 4 个木棍 - **工作台...
    score=-3.6879  ## 活塞  活塞是红石驱动的机械元件，可以推动方块。  ### 普通活塞  **合成配方**：3 个木板 + 4 个圆石 + 1 个铁锭 + 1 个红石粉  ...
    score=-3.8315  ## 漏斗  漏斗是用于传输物品的红石元件，可以朝下或侧面输送物品。  ### 合成配方  5 个铁锭 + 1 个箱子（铁锭摆 V 字型，底部中间放箱子）...
    score=-4.6205  ## 锻造台与下界合金装备升级  锻造台是用于将钻石装备升级为下界合金装备的工作站方块。  ### 锻造台  **合成配方**：4 个木板 + 2 个铁锭  *...
    score=-5.4455  ### 常见烧炼配方  | 输入 | 输出 | 用途 | |------|------|------| | 铁矿石 | 铁锭 | 制作工具、盔甲 | | 金矿石...
    score=-5.6500  ### 经验获取方式  **采矿**：挖矿（尤其是煤矿、红石矿、青金石矿、下界石英矿）会掉落经验球  **刷怪**：击杀生物会掉落经验，Boss 掉落最多： -...
    score=-5.7584  ## 剑  剑是 Minecraft 中的主要近战武器。  ### 